In [5]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

# BB84 Quantum Key Distribution — With Attacker (Eve)

This notebook extends the plain BB84 simulation by inserting **Eve** (an eavesdropper) between Alice and Bob.

### Eve's intercept-resend attack
Eve intercepts each qubit, picks a random basis (using quantum randomness), measures it, and forwards a **freshly prepared** qubit encoding her measurement result to Bob.  
Because Eve often chooses the wrong basis, she disturbs ~25% of the sifted key bits, which Alice and Bob can detect during error checking.

### Detection
Alice and Bob sacrifice a sample of their sifted key to estimate the error rate.  
An error rate above the threshold (10%) triggers an attack warning and the key exchange is aborted.

In [6]:
%pip install qiskit==1.2.4 qiskit-aer==0.15.1 pylatexenc==2.10 -q

In [7]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
import math

# ── Shared simulator ────────────────────────────────────────────────────────
simulator = AerSimulator()

# ─────────────────────────────────────────────────────────────────────────────
# UTILITY: quantum random-bit generator
# Measures n qubits each prepared in |+⟩ = H|0⟩ to get n unbiased random bits.
# ─────────────────────────────────────────────────────────────────────────────
def quantum_random_bits(n: int) -> list[int]:
    """Return a list of n random bits by measuring qubits in the |+⟩ state.
    Batches requests to stay within the simulator's 29-qubit limit."""
    MAX_BATCH = 29
    bits = []
    remaining = n
    while remaining > 0:
        batch = min(remaining, MAX_BATCH)
        qc = QuantumCircuit(batch, batch)
        qc.h(range(batch))
        qc.measure(range(batch), range(batch))
        job = simulator.run(transpile(qc, simulator), shots=1, memory=True)
        result_str = job.result().get_memory()[0]
        bits.extend(int(b) for b in reversed(result_str))
        remaining -= batch
    return bits[:n]

print("Utility ready. Sample of 8 quantum random bits:", quantum_random_bits(8))

Utility ready. Sample of 8 quantum random bits: [1, 1, 0, 0, 1, 1, 0, 0]


In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# ALICE — encoding
# Basis 0 → rectilinear (+);  Basis 1 → diagonal (×)
#   basis 0, bit 0 → |0⟩
#   basis 0, bit 1 → |1⟩
#   basis 1, bit 0 → |+⟩  (H|0⟩)
#   basis 1, bit 1 → |−⟩  (H|1⟩)
# ─────────────────────────────────────────────────────────────────────────────
N_QUBITS = 100
SAMPLE_FRACTION = 0.2
DETECTION_THRESHOLD = 0.10

def alice_encode(bits: list[int], bases: list[int]) -> list[QuantumCircuit]:
    """Alice encodes each bit in the chosen basis. Returns single-qubit circuits (no measurement)."""
    circuits = []
    for bit, basis in zip(bits, bases):
        qc = QuantumCircuit(1, 1)
        if bit == 1:
            qc.x(0)
        if basis == 1:
            qc.h(0)
        circuits.append(qc)
    return circuits

alice_bits  = quantum_random_bits(N_QUBITS)
alice_bases = quantum_random_bits(N_QUBITS)
alice_circuits = alice_encode(alice_bits, alice_bases)

print(f"Alice prepared {N_QUBITS} qubits.")
print(f"Alice bits  (first 20): {alice_bits[:20]}")
print(f"Alice bases (first 20): {alice_bases[:20]}  (0=+, 1=×)")

Alice prepared 100 qubits.
Alice bits  (first 20): [0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1]
Alice bases (first 20): [0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1]  (0=+, 1=×)


In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# EVE — intercept-resend attack
#
# For each qubit Alice sends, Eve:
#   1. Chooses a random basis (using quantum randomness — H|0⟩ measurement).
#   2. Measures the qubit in that basis (this collapses/destroys the original state).
#   3. Prepares a NEW qubit encoding her measurement result in her chosen basis.
#   4. Forwards that new qubit to Bob.
#
# When Eve's basis matches Alice's, she gets the right bit and Bob sees no error.
# When they differ, Eve gets a random result → 50% chance Bob's bit is wrong.
# Overall ~25% of sifted bits will be disrupted.
# ─────────────────────────────────────────────────────────────────────────────
def eve_intercept(circuits: list[QuantumCircuit]) -> tuple[list[QuantumCircuit], list[int], list[int]]:
    """
    Eve intercepts all qubits.
    Returns:
      - tampered_circuits : the qubits Eve forwards to Bob
      - eve_bases         : the bases Eve used (kept secret from Alice/Bob)
      - eve_bits          : the bits Eve measured (her partial key info)
    """
    n = len(circuits)
    eve_bases = quantum_random_bits(n)   # Eve's random basis choices
    eve_bits  = []
    tampered  = []

    for qc, basis in zip(circuits, eve_bases):
        # Step 1 & 2: Eve measures in her chosen basis
        intercept_qc = qc.copy()
        if basis == 1:
            intercept_qc.h(0)          # rotate to diagonal before measuring
        intercept_qc.measure(0, 0)
        job = simulator.run(transpile(intercept_qc, simulator), shots=1, memory=True)
        measured_bit = int(job.result().get_memory()[0])
        eve_bits.append(measured_bit)

        # Step 3 & 4: Eve re-encodes the measured bit and sends to Bob
        resend_qc = QuantumCircuit(1, 1)
        if measured_bit == 1:
            resend_qc.x(0)
        if basis == 1:
            resend_qc.h(0)
        tampered.append(resend_qc)

    return tampered, eve_bases, eve_bits

tampered_circuits, eve_bases, eve_bits = eve_intercept(alice_circuits)

print("Eve has intercepted all qubits and forwarded tampered versions to Bob.")
print(f"Eve bases (first 20): {eve_bases[:20]}")
print(f"Eve bits  (first 20): {eve_bits[:20]}")

Eve has intercepted all qubits and forwarded tampered versions to Bob.
Eve bases (first 20): [0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0]
Eve bits  (first 20): [0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1]


In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# BOB — measurement (Bob receives Eve's tampered qubits, unaware of the attack)
# ─────────────────────────────────────────────────────────────────────────────
def bob_measure(circuits: list[QuantumCircuit], bases: list[int]) -> list[int]:
    """Bob measures each qubit in his randomly chosen basis."""
    results = []
    for qc, basis in zip(circuits, bases):
        meas_qc = qc.copy()
        if basis == 1:
            meas_qc.h(0)
        meas_qc.measure(0, 0)
        job = simulator.run(transpile(meas_qc, simulator), shots=1, memory=True)
        results.append(int(job.result().get_memory()[0]))
    return results

bob_bases = quantum_random_bits(N_QUBITS)
bob_bits  = bob_measure(tampered_circuits, bob_bases)   # Bob gets Eve's qubits!

print(f"Bob measured {N_QUBITS} qubits (unaware they were tampered with).")
print(f"Bob bases (first 20): {bob_bases[:20]}")
print(f"Bob bits  (first 20): {bob_bits[:20]}")

Bob measured 100 qubits (unaware they were tampered with).
Bob bases (first 20): [0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1]
Bob bits  (first 20): [0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1]


In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# SIFTING — Alice and Bob publicly compare bases
# ─────────────────────────────────────────────────────────────────────────────
def sift_key(alice_bases, bob_bases, alice_bits, bob_bits):
    alice_sifted, bob_sifted, positions = [], [], []
    for i, (ab, bb) in enumerate(zip(alice_bases, bob_bases)):
        if ab == bb:
            alice_sifted.append(alice_bits[i])
            bob_sifted.append(bob_bits[i])
            positions.append(i)
    return alice_sifted, bob_sifted, positions

alice_sifted, bob_sifted, matching_positions = sift_key(
    alice_bases, bob_bases, alice_bits, bob_bits
)

print(f"Sifted key length: {len(alice_sifted)}  (expected ≈ {N_QUBITS//2})")
print(f"Alice sifted (first 20): {alice_sifted[:20]}")
print(f"Bob   sifted (first 20): {bob_sifted[:20]}")

Sifted key length: 55  (expected ≈ 50)
Alice sifted (first 20): [0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1]
Bob   sifted (first 20): [0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1]


In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# ERROR CHECKING — detecting Eve
#
# Alice and Bob publicly compare a sample of their sifted bits.
# Eve's intercept-resend attack introduces ≈25% errors in the sifted key,
# well above the 10% detection threshold.
# ─────────────────────────────────────────────────────────────────────────────
sample_size = max(1, int(len(alice_sifted) * SAMPLE_FRACTION))
sample_alice = alice_sifted[:sample_size]
sample_bob   = bob_sifted[:sample_size]

errors     = sum(a != b for a, b in zip(sample_alice, sample_bob))
error_rate = errors / sample_size

# ── Eve's knowledge (for illustration only — not available to Alice/Bob) ────
sifted_positions_set = set(matching_positions)
eve_correct = sum(
    1 for i in matching_positions
    if eve_bases[i] == alice_bases[i]
)
eve_knowledge = eve_correct / len(matching_positions) if matching_positions else 0

print("── Error checking ──────────────────────────────────")
print(f"  Sample size          : {sample_size} bits")
print(f"  Errors found         : {errors}")
print(f"  Error rate           : {error_rate:.1%}  (expected ≈25% with full intercept)")
print(f"  Threshold            : {DETECTION_THRESHOLD:.0%}")
print()
print(f"  [Eve's perspective]  : Eve guessed the right basis {eve_knowledge:.1%} of the time")
print(f"                         (expected ≈50% — she gains ~50% of sifted key but causes ~25% errors)")
print()

if error_rate > DETECTION_THRESHOLD:
    print("  ⚠️  ATTACK DETECTED — error rate too high. Aborting key exchange!")
    print("      Alice and Bob discard all key material and will retry over a secure channel.")
else:
    # This branch is very unlikely with a full intercept-resend attack (~0% probability)
    print("  ✅ No attack detected (Eve got lucky with her basis choices).")
    final_key = alice_sifted[sample_size:]
    print(f"  Final key: {final_key[:20]}...")

── Error checking ──────────────────────────────────
  Sample size          : 11 bits
  Errors found         : 1
  Error rate           : 9.1%  (expected ≈25% with full intercept)
  Threshold            : 10%

  [Eve's perspective]  : Eve guessed the right basis 52.7% of the time
                         (expected ≈50% — she gains ~50% of sifted key but causes ~25% errors)

  ✅ No attack detected (Eve got lucky with her basis choices).
  Final key: [1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1]...


## Discussion

### Why does Eve cause errors?
When Eve measures a qubit in the **wrong basis**, she collapses the quantum state to a random eigenstate of her basis. The qubit she forwards to Bob is then in the wrong state with probability 1/2. Since Eve guesses the wrong basis 50% of the time, approximately **25%** of the sifted key bits will be corrupted.

This is a fundamental consequence of the **no-cloning theorem**: Eve cannot copy an unknown quantum state; she must measure and destroy it, inevitably disturbing it.

### Partial attack variant
A subtler Eve could intercept only a fraction of qubits, reducing the error rate below the detection threshold — but also gaining less key information. This is the classic security trade-off in BB84. In practice, information-theoretic security proofs bound the key information Eve can gain for any observed error rate, enabling **privacy amplification** to distil a secure key.

### Threshold choice
The 10% threshold used here is conservative; BB84 is theoretically secure up to an error rate of about **11%** (the quantum bit error rate threshold). Above that, the channel noise alone could explain the errors.